# [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)

## Objective

The goal in this competition is to take an image of a handwritten single digit, and determine what that digit is. For every in the test set, model should predict the correct label.

## [Dataset](https://www.kaggle.com/competitions/digit-recognizer/data)

The data files train.csv and test.csv contain gray-scale images of hand-drawn digits, from zero through nine.

Each image is 28 pixels in height and 28 pixels in width, for a total of 784 pixels in total. Each pixel has a single pixel-value associated with it, indicating the lightness or darkness of that pixel, with higher numbers meaning darker. This pixel-value is an integer between 0 and 255, inclusive.

The training data set, (train.csv), has 785 columns. The first column, called "label", is the digit that was drawn by the user. The rest of the columns contain the pixel-values of the associated image.

Each pixel column in the training set has a name like pixelx, where x is an integer between 0 and 783, inclusive. To locate this pixel on the image, suppose that we have decomposed x as x = i * 28 + j, where i and j are integers between 0 and 27, inclusive. Then pixelx is located on row i and column j of a 28 x 28 matrix, (indexing by zero).

## Evaluation

Evaluated on the categorization accuracy of model predictions (the percentage of images model predict correct). Accuracy should be above 97 %

## Data Loading

In [ ]:
from pathlib import Path

import pandas as pd

KAGLE_DATA_DIR = Path("/kaggle/input/digit-recognizer")

TRAIN_DATA_DIR = KAGLE_DATA_DIR / "train.csv"
TEST_DATA_DIR = KAGLE_DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_DATA_DIR)
submitted_df = pd.read_csv(TEST_DATA_DIR)

In [ ]:
train_df.head()

In [ ]:
len(train_df)

In [ ]:
submitted_df.head()

In [ ]:
pixels = train_df.drop(columns=["label"])
labels = train_df["label"]

pixels.head()

In [ ]:
labels.head()

In [ ]:
labels.value_counts().sort_index()

## Exploratory Data Analysis

In [ ]:
from matplotlib import pyplot as plt

for i in range(20):
    plt.subplot(5,4, i+1)
    plt.tight_layout()
    plt.imshow(pixels.iloc[i].values.reshape(28, 28), cmap=plt.cm.binary)
    plt.title('Digit: {}'.format(labels.iloc[i]))
    plt.xticks([])
    plt.yticks([])

plt.show()

## ETL (Extract, Transform and Load)

In [ ]:
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

IMG_WIDTH = 28
IMG_HEIGHT = IMG_WIDTH
IMG_CHANNEL = 1

MAX_RGB_NO = 255.

BATCH_SIZE = 32

random_state = 42
random_seed = random_state

reshape_dim = (-1, IMG_CHANNEL, IMG_WIDTH, IMG_HEIGHT)

torch.manual_seed(42)

label_encoder = LabelBinarizer()

X_train, X_test, y_train, y_test = train_test_split(pixels, labels, test_size=0.1, random_state=random_state, stratify=labels)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=random_state, stratify=y_train)

X_train = np.reshape((X_train / MAX_RGB_NO).to_numpy().astype(np.float32), reshape_dim)
y_train = label_encoder.fit_transform(y_train).astype(np.float32)
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

X_valid = np.reshape((X_valid / MAX_RGB_NO).to_numpy().astype(np.float32), reshape_dim)
y_valid = label_encoder.transform(y_valid).astype(np.float32)
valid_dataset = TensorDataset(torch.from_numpy(X_valid), torch.from_numpy(y_valid))
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

X_test = np.reshape((X_test / MAX_RGB_NO).to_numpy().astype(np.float32), reshape_dim)
y_test = label_encoder.transform(y_test).astype(np.float32)
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

## Model Creation

In [ ]:
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class LeNet5Relu(nn.Module):
    def __init__(self, num_classes: int):
        super(LeNet5Relu, self).__init__()

        self.conv2d_1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2)
        self.conv2d_2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.conv2d_3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1, padding=0)
        
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, num_classes)
        
        self.relu = nn.ReLU()
        self.avgpool2d = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Layer 1
        out = self.relu(self.conv2d_1(x))
        out = self.avgpool2d(out) # 6x28x28 -> 6x14x14

        # Layer 2
        out = self.relu(self.conv2d_2(out))
        out = self.avgpool2d(out) # 16x10x10 -> 16x5x5

        # Layer 3
        out = self.relu(self.conv2d_3(out)) # 120x1x1

        # Flatten the output for the fully connected layers
        out = torch.flatten(out, start_dim = 1)

        # FC layer 1
        out = self.relu(self.fc1(out))

        # FC layer 2
        out = self.relu(self.fc2(out))

        return out


In [ ]:
from torchsummary import summary

input_size = (IMG_CHANNEL, IMG_WIDTH, IMG_HEIGHT)

num_classes = len(list(labels.unique()))

model = LeNet5Relu(num_classes).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params = model.parameters())

summary(model, input_size=input_size) # (channels, height, width)

## Model Training

In [ ]:
epochs = 30

for epoch in range(epochs):
    total_loss_train = 0.
    total_loss_val = 0.
    total_acc_train = 0.
    total_acc_val = 0
    
    model.train()

    for inputs, targets in train_dataloader:
        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss_train = loss_fn(outputs, targets)

        total_loss_train += loss_train.item()
        
        loss_train.backward()

        acc_train = (torch.argmax(outputs, dim=1) == torch.argmax(targets, dim=1)).sum().item()

        total_acc_train += acc_train
    
        optimizer.step()

    model.eval()

    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
        for inputs, targets in valid_dataloader:
            outputs = model(inputs)
            loss_val = loss_fn(outputs, targets)

            total_loss_val += loss_val.item()

            acc_val = (torch.argmax(outputs, dim=1) == torch.argmax(targets, dim=1)).sum().item()
            
            total_acc_val += acc_val

    acc_train_percent = round(total_acc_train / len(train_dataset) * 100, 2)
    acc_val_percent = round(total_acc_val / len(valid_dataset) * 100, 2)
    
    print(f'Epoch {epoch + 1}/{epochs} | Train loss: {total_loss_train / 1000: .4f} | Train accuracy: {acc_train_percent:.2f}% | Validation loss: {total_loss_val / 1000:.4f} | Validation accuracy: {acc_val_percent:.2f}%')

## Model Evaluation

In [ ]:
total_loss_test = 0.
total_acc_test = 0.

with torch.no_grad():
    for inputs, targets in test_dataloader:
        outputs = model(inputs)
        loss_test = loss_fn(outputs, targets)

        total_loss_test += loss_test.item()

        acc_test = (torch.argmax(outputs, dim=1) == torch.argmax(targets, dim=1)).sum().item()
        
        total_acc_test += acc_test

acc_test_percent = round(total_acc_test / len(test_dataset) * 100, 2)

print(f'Test loss: {total_loss_test / 1000: .4f} | Test accuracy: {acc_test_percent:.2f}%')

## Submission

In [ ]:
features = np.reshape((submitted_df / MAX_RGB_NO).to_numpy().astype(np.float32), reshape_dim)

features_dataloader = DataLoader(torch.from_numpy(features), batch_size=BATCH_SIZE, shuffle=False)

preds = []

with torch.no_grad():
    for inputs in features_dataloader:
        outputs = model(inputs)

        preds.extend(torch.argmax(outputs, dim=1).tolist())

In [ ]:
submit_df = pd.read_csv(KAGLE_DATA_DIR / "sample_submission.csv")

submit_df.loc[:, "Label"] = preds

submit_df.to_csv("submission.csv", index=False)

print("DONE")